# 하이브리드 검색과 Reranker로 Retriever 성능 높이기
- 지금까지는 문서를 로드하고, 청크로 나누고, 임베딩한 뒤 Chroma retriever 로 검색했습니다.
- 이번에는 **BM25 키워드 검색**, **Chroma 벡터 검색**, **EnsembleRetriever 하이브리드 검색**, **Reranker** 를 붙여 검색 품질을 높입니다.
- 핵심 목표는 "1차로 넓게 후보를 찾고, 2차로 질문에 더 맞는 후보를 위로 올리는 것"입니다.


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [18]:
# 필요한 라이브러리 설치
# uv add -qU rank-bm25 langchain-classic langchain-cohere kiwipiepy langchain-chroma langchain-openai python-dotenv

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
OPENAI_API_KEY=sk-...
COHERE_API_KEY=...
```

- `OPENAI_API_KEY`: 임베딩, LLM-as-reranker, RAG 답변 생성에 사용
- `COHERE_API_KEY`: Cohere Rerank 실습에 사용. 없으면 해당 셀은 건너뜁니다.

In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 2. 회사 문서 로드 + 청크 분할

- 이전 RAG 실습과 같은 `data/company_docs` 폴더 사용
- 여러 문서를 한 번에 로드한 뒤, 검색에 적합한 크기로 청킹

In [20]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("data2/company_docs")

In [21]:
def load_text_documents(data_dir: Path):
    all_docs = []
    for pattern in ["**/*.txt", "**/*.md"]:
        loader = DirectoryLoader(
            str(data_dir),
            glob=pattern,
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        )
        all_docs.extend(loader.load())
    return all_docs

raw_docs = load_text_documents(DATA_DIR)

for doc in raw_docs:
    source = doc.metadata.get("source", "")
    doc.metadata["category"] = Path(source).stem

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""],
)
split_docs = text_splitter.split_documents(raw_docs)

for i, doc in enumerate(split_docs):
    doc.metadata["doc_id"] = f"company-doc-{i:04d}"

In [23]:
print(f"원본 문서 수: {len(raw_docs)}")
print(f"검색용 청크 수: {len(split_docs)}")
print("첫 번째 청크 metadata:", split_docs[0].metadata)
print(split_docs[0].page_content[:300])

원본 문서 수: 7
검색용 청크 수: 69
첫 번째 청크 metadata: {'source': 'data2\\company_docs\\hr_policy.txt', 'category': 'hr_policy', 'doc_id': 'company-doc-0000'}
# [인사 정책 매뉴얼] 즐겁고 공정한 직장 문화를 위한 가이드

본 안내서는 우리 회사의 핵심 인사 원칙과 규정을 담고 있습니다. 모든 임직원은 본 정책을 준수하며, 상호 존중과 성과 중심의 문화를 함께 만들어 갑니다.

---

## 1. 휴가 및 근태 정책

우리 회사는 임직원의 충분한 휴식과 일과 삶의 균형(Work-Life Balance)을 전폭적으로 지원합니다.

### 1.1 연차 유급 휴가

* **발생 기준:** * **신입사원:** 입사 1년 미만 시, 1개월 개근 시 1일씩 발생 (최대 11일).
* **정기 연


## 3. Chroma 벡터 검색 준비

- 벡터 검색은 질문과 문서의 **의미 유사도** 를 잘 잡음
- 대신 고유명사, 코드, 정확한 키워드 매칭에는 약할 수 있음

In [24]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

collection_name = "company_docs_hybrid_rerank"

# 반복 실행 대비: 같은 collection이 있으면 지우고 다시 생성
reset_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)
reset_store.delete_collection()

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    ids=[doc.metadata["doc_id"] for doc in split_docs],
    collection_name=collection_name,
)

In [25]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("저장 문서 수:", vectorstore._collection.count())

저장 문서 수: 69


## 4. BM25 키워드 검색 준비

- 한국어는 공백만으로 단어를 나누면 조사와 어미 때문에 BM25 품질이 흔들릴 수 있기 때문에, 가능하면 `kiwipiepy` 같은 형태소 분석기를 붙임


| 검색 방식 | 잘하는 것 | 약한 것 |
|---|---|---|
| **BM25** | 고유명사, 코드, 정확한 단어 매칭 | 동의어, 표현 변형, 의미 유사도 |
| **벡터 검색** | 동의어, 문맥, 자연어 의미 | 최신 고유명사, 약어, 정확한 문자열 |



In [26]:
#uv add rank_bm25 kiwipiepy

In [27]:
from langchain_community.retrievers import BM25Retriever

# 한국어 형태소 분석기
from kiwipiepy import Kiwi

kiwi = Kiwi()

def korean_tokenizer(text: str) -> list[str]:
    """명사, 동사, 형용사, 외국어, 숫자 중심으로 BM25 토큰을 만듭니다."""
    meaningful_tags = {"NNG", "NNP", "VV", "VA", "SL", "SN"} # 품사: 일반명사(NNG), 고유명사(NNP), 동사(VV), 형용사(VA), 알파벳(SL) 
    return [
        token.form.lower()
        for token in kiwi.tokenize(text)
        if token.tag in meaningful_tags
    ]


sample = "재택근무 신청은 어디에서 승인받아야 하나요?"
print("예시 토큰:", korean_tokenizer(sample))

예시 토큰: ['재택근무', '신청', '승인']


In [28]:
bm25_retriever = BM25Retriever.from_documents(
    split_docs,
    preprocess_func=korean_tokenizer,       # preprocess_func에 한국어 형태소 토크나이저를 넣어, BM25가 조사/어미보다 의미 있는 단어 중심으로 검색하도록 함.
    k = 5
)

## 5. BM25 검색과 벡터 검색 비교

- BM25는 정확한 단어가 있는 문서를 잘 찾고, 벡터 검색은 표현이 조금 달라도 의미가 가까운 문서를 잘 찾음

In [29]:
from langchain_core.documents import Document


def print_docs(title: str, docs: list[Document], max_chars: int = 220):
    print(f"\n=== {title} ({len(docs)}개) ===")
    for i, doc in enumerate(docs, start=1):
        category = doc.metadata.get("category", "unknown")
        text = doc.page_content.replace("\n", " ")
        print(f"\n[{i}] category={category}")
        print(text[:max_chars])


question = '법인카드 영수증은 언제까지 제출해야 하나요?'

bm25_results = bm25_retriever.invoke(question)
vector_results = vector_retriever.invoke(question)

print(f"질문: {question}")
print_docs("BM25 결과", bm25_results)
print_docs("벡터 결과", vector_results)

질문: 법인카드 영수증은 언제까지 제출해야 하나요?

=== BM25 결과 (5개) ===

[1] category=expense_policy
# 경비 처리 규정  본 규정은 회사 업무 수행에 필요한 경비 처리 및 정산 절차를 안내합니다. 모든 경비는 투명하고 합리적으로 집행되어야 합니다.  ---  ## 1. 경비 처리 원칙  ### 1.1 기본 원칙 - **업무 관련성:** 회사 업무 수행에 직접 관련된 비용만 처리 - **적정성:** 사회 통념상 적정한 수준의 금액 - **증빙 필수:** 모든 경비는 적격 증빙 첨부 필수 - 

[2] category=expense_policy
---  ## 8. 위반 시 제재  | 위반 유형 | 제재 | |----------|------| | 개인 용도 사용 | 전액 환수 + 경고 | | 허위 증빙 제출 | 전액 환수 + 징계 | | 반복적 규정 위반 | 법인카드 회수 + 인사 조치 | | 금지 업종 사용 | 전액 환수 + 징계위원회 회부 |  ---  > **문의처** > 경비 처리 관련 문의는 재무팀으로 연락 바랍니다. > -

[3] category=hr_policy
---  ## 5. 퇴직 및 오프보딩 절차  마지막까지 아름다운 마무리를 위해 원활한 인수인계를 지원합니다.  ### 5.1 퇴직 절차  * **사직서 제출:** 퇴직 희망일 기준 최소 **1개월 전**에 사직서를 제출하여 인력 충원 및 업무 승계 시간을 확보해야 합니다. * **인수인계:** 퇴직 전 '업무 인수인계서'를 작성하여 팀장 및 후임자에게 공유하고, 사용하던 법인 자산(노트북, 

[4] category=expense_policy
### 6.2 구매 절차 1. ERP 구매 요청 등록 2. 승인권자 승인 3. 총무팀 구매 진행 (또는 직접 구매) 4. 납품 확인 및 검수 5. 정산 처리  ---  ## 7. 정산 및 지급  ### 7.1 정산 마감 - **매주 금요일** 17:00까지 신청 건 → 익주 수요일 지급 - 월말 마감: **매월 

## 6. 하이브리드 검색, BM25 + 벡터 동시에

- `EnsembleRetriever` 는 여러 retriever 의 결과를 RRF(Reciprocal Rank Fusion) 방식으로 합침
- RRF는 각 retriever 의 원점수를 직접 섞지 않고, **순위**를 기준으로 합침
- 즉, RRF 는 각 retriever 가 매긴 순위 (1등, 2등, ...) 를 받아서 "여러 retriever 에서 모두 상위에 든 문서" 가 가장 높은 점수를 받게 함.
- 그래서 BM25 점수와 벡터 유사도 점수처럼 스케일이 다른 검색 결과도 비교적 안정적으로 결합할 수 있음

| 방식 | 강점 | 약점 |
|------|------|------|
| **BM25 (키워드)** | 정확한 고유명사 매칭 | 의미 유사성 약함 |
| **Chroma (의미)** | 문맥 이해 | 정확한 키워드 매칭 약함 |
| **Hybrid** | 두 방식의 장점 결합 | 가중치 튜닝 필요 |

In [30]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]              # [BM25, Vector]    # 두 retriever의 RRF 점수 가중치
)

In [31]:
for q in [
    "재택근무 신청 방법",                      # 의미 검색 강점
    "법인카드 영수증 제출 기한",                # 키워드 강점
    "API 키는 어디에 보관해야 해?",             # 둘다 필요 
]:
    print(f"\n질문: {q}")
    print(f"한국어 토큰: {korean_tokenizer(q)}")
    print_docs("Hybrid 결과", hybrid_retriever.invoke(q), max_chars=180)


질문: 재택근무 신청 방법
한국어 토큰: ['재택근무', '신청', '방법']

=== Hybrid 결과 (7개) ===

[1] category=faq
---  ## 2. 인사(HR) 및 복무 규정  ### Q4. 연차 및 특별 휴가 관리  * **잔여일 확인:** [HR Portal] → [내 정보] → [휴가/근태]에서 실시간 확인이 가능합니다. * **신청 기한:** 연차는 최소 3일 전 신청을 권장하며, 당일 급차 사용 시에는 팀장님께 메신저 또는 유선 보고 후 

[2] category=onboarding_guide
---  ## 6. 주요 연락처  ### 6.1 지원 부서  | 부서 | 담당 업무 | 연락처 | |------|----------|--------| | **인사팀** | 근태, 복리후생, 급여 | hr@company.com / 내선 9012 | | **IT지원팀** | 장비, 계정, 시스템 | it-support@com

[3] category=onboarding_guide
### Q2: 재택근무는 어떻게 신청하나요? HR Portal에서 신청하며, 주 최대 2회까지 가능합니다. 팀장 사전 승인 필요.  ### Q3: 점심시간은 어떻게 되나요? 12:00-13:00이며, 구내식당 또는 외부 식당 이용 가능합니다.  ### Q4: 야근 시 저녁 식대가 지원되나요? 19시 이후 야근 시 1만 원

[4] category=faq
---  ## 5. 시설 이용 및 기타 지원  ### Q12. 주차 등록 및 차량 관리  * **신규 등록:** 차량등록증 사본을 지참하여 총무팀 방문. * **주차 요금:** 월 30,000원 (급여 공제), 전기차 충전 구역 이용 시 충전비 별도. * **방문객 주차:** 인트라넷에서 '방문객 주차 예약'을 선행해야 

[5] category=expense_policy
### 6.2 구매 절차 1. ERP 구매 요청 등록 2. 승인권자 승인 3. 총무팀 구매 진행 (또는 직접 구매) 4. 납품 확인 및 검수 5

## 7. 가중치 실험

- 하이브리드 검색은 도메인에 따라 BM25와 벡터의 비율 조정

| 도메인 | 출발 가중치 |
|---|---|
| 기술 문서, 코드, 고유명사 많음 | `[0.6, 0.4]` 또는 `[0.7, 0.3]` |
| 자연어 Q&A, 표현 다양함 | `[0.3, 0.7]` 또는 `[0.4, 0.6]` |
| 처음 시작할 때 | `[0.5, 0.5]` |

In [32]:
weight_sets = {
    "BM25 only": [1.0, 0.0],
    "BM25 우선": [0.7, 0.3],
    "균형": [0.5, 0.5],
    "Vector 우선": [0.3, 0.7],
    "Vector only": [0.0, 1.0],
}

question = "신규 입사자 보안 교육은 언제까지 받아야 하나요?"

for label, weights in weight_sets.items():
    retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights = weights)
    


    docs = retriever.invoke(question)
    top = docs[0]
    print(f"\n[{label}] weights={weights}")
    print(f"top category={top.metadata.get('category')}")
    print(top.page_content.replace("\n", " ")[:220])


[BM25 only] weights=[1.0, 0.0]
top category=faq
---  ## 5. 시설 이용 및 기타 지원  ### Q12. 주차 등록 및 차량 관리  * **신규 등록:** 차량등록증 사본을 지참하여 총무팀 방문. * **주차 요금:** 월 30,000원 (급여 공제), 전기차 충전 구역 이용 시 충전비 별도. * **방문객 주차:** 인트라넷에서 '방문객 주차 예약'을 선행해야 2시간 무료 주차가 지원됩니다.  ### Q13. 명함 신청 및 제작  

[BM25 우선] weights=[0.7, 0.3]
top category=security_guide
---  ## 8. 보안 교육  ### 8.1 필수 교육 - **신입사원:** 입사 1주 내 보안 기초 교육 이수 - **전 직원:** 연 1회 보안 인식 교육 (2시간) - **개발자:** 시큐어 코딩 교육 (연 1회) - **관리자:** 보안 관리자 심화 교육 (연 1회)  ### 8.2 모의 훈련 - 분기별 피싱 메일 모의 훈련 - 연 1회 침해사고 대응 훈련 - 훈련 결과에 따른 추가

[균형] weights=[0.5, 0.5]
top category=security_guide
---  ## 8. 보안 교육  ### 8.1 필수 교육 - **신입사원:** 입사 1주 내 보안 기초 교육 이수 - **전 직원:** 연 1회 보안 인식 교육 (2시간) - **개발자:** 시큐어 코딩 교육 (연 1회) - **관리자:** 보안 관리자 심화 교육 (연 1회)  ### 8.2 모의 훈련 - 분기별 피싱 메일 모의 훈련 - 연 1회 침해사고 대응 훈련 - 훈련 결과에 따른 추가

[Vector 우선] weights=[0.3, 0.7]
top category=security_guide
---  ## 8. 보안 교육  ### 8.1 필수 교육 - **신입사원:** 입사 1주 내 보안 기초 교육 이수 - **전 직원:** 연 1회 보안 인식 교육 (2시간) - **개발자:** 시큐어 코딩 교육 

## 8. Reranker, 후보를 한 번 더 정렬

- 하이브리드 검색은 1차 후보를 넓게 찾는 역할
- Reranker는 질문과 후보 문서를 **쌍으로 직접 비교**해서 더 관련 있는 문서를 위로 올림

- 운영에서 자주 쓰는 패턴:

```text
1차 retriever: BM25 + Vector 로 후보 20~30개 확보
2차 reranker: 후보를 3~5개로 재정렬/압축
최종 LLM: 재정렬된 문서만 참고해 답변
```

In [33]:
# uv add langchain_cohere

In [34]:
load_dotenv()

print("COHERE_API_KEY:", "있음" if os.getenv("COHERE_API_KEY") else "없음")

COHERE_API_KEY: 있음


In [35]:
# reranker가 만들어지지 않는 경우를 대비해 기본값을 None으로 둠
rerank_retriever = None

# Cohere에서 제공하는 reranker 모델
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

# 하이브리드 검색 결과 중 질문과 가장 관련 높은 상위 3개만 남기도록 설정
# 필요 시 V4(rerank-v4.0-pro)도 공식 샘플에 보이니 교체 가능. (2026-06기준)
compressor = CohereRerank(model="rerank-v3.5", top_n=3)

# 1차 검색기(hybrid_retriever)의 결과를 Cohere reranker로 재정렬하는 retriever 생성
rerank_retriever=ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=hybrid_retriever
)

question = "장애 보고서에는 어떤 내용이 들어가야 하나요?"

print(f"질문: {question}")

print_docs("Hybrid 상위 후보", hybrid_retriever.invoke(question), max_chars=300)
print_docs("Cohere Rerank 결과", rerank_retriever.invoke(question) , max_chars=300)


질문: 장애 보고서에는 어떤 내용이 들어가야 하나요?

=== Hybrid 상위 후보 (10개) ===

[1] category=product_manual
---  ## 6. 실시간 소통 및 화상회의  ### 6.1 채팅 채널 전략  - **스레드(Thread) 사용:** 특정 메시지에 대한 답변은 '스레드'로 작성하여 채팅창이 지저분해지는 것을 방지하세요. - **멘션(@):** 급한 호출은 `@이름`, 전체 공지는 `@channel`을 활용하세요. 방해 금지 모드 중인 사용자에게는 알림이 가지 않도록 배려할 수 있습니다.  ### 6.2 화상회의 편의 기능  - **화면 공유:** 특정 프로그램 창만 공유하거나 소리까지 포함하여 공유할 수 있습니다. - **화이트보드:** 회의 중

[2] category=dev_standards
**배포 후:** - [ ] 헬스 체크 확인 - [ ] 핵심 기능 스모크 테스트 - [ ] 에러 모니터링 (Sentry, DataDog) - [ ] 성능 메트릭 확인  ### 5.4 롤백 절차  1. 문제 감지 시 즉시 PagerDuty 알림 2. 이전 버전으로 롤백 (1-Click Rollback) 3. 인시던트 채널 생성 및 팀 소집 4. 원인 분석 및 수정 5. 포스트모템 작성  ---  ## 6. 문서화  ### 6.1 필수 문서  | 문서 | 위치 | 담당 | |------|------|------| | README.md 

[3] category=onboarding_guide
### 3.3 Day 3 (수요일) | 시간 | 내용 | 담당 | |------|------|------| | 09:00-12:00 | 제품/서비스 이해 | 제품팀 | | 13:00-17:00 | 업무 프로세스 학습 | 멘토 |  ### 3.4 Day 4 (목요일) | 시간 | 내용 | 담당 | |------|------|------| | 09:00-12:00 | 협업 도구 실습 | IT지원팀 | | 13:00-17:00 | 첫 업무 배정 및 시작

## 9. LLM-as-reranker 대안

- Cohere 같은 전용 reranker가 없을 때는 LLM에게 "질문과 문서의 관련도를 0~10점으로 평가"하게 만들 수 있음
- 비용·속도는 떨어지지만 외부 키 없이 됨

In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model = 'gpt-5.4-mini',
                 temperature=0)

SCORE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "문서가 질문에 얼마나 관련 있는지 0~10점으로만 답해. 숫자만"),
    ("user", "질문: {question}\n\n문서: {doc}"),
])


scorer = SCORE_PROMPT | llm | StrOutputParser()


def llm_rerank(question: str, candidates: list[Document], top_n: int = 3):
    scored = []
    for d in candidates:
        try:
            score = float(scorer.invoke({"question": question, "doc": d.page_content}).strip())
        except ValueError:
            score = 0.0
        scored.append((d, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


In [42]:
question = '재택근무 신청은 어떤 절차로 하나요?'
candidates = hybrid_retriever.invoke(question)


print_docs("Hybrid 후보", candidates, max_chars=150)

reranked = llm_rerank(question, candidates, top_n=3)
if reranked:
    print("\n=== LLM Rerank 결과 ===")
    for i, (doc, score) in enumerate(reranked, start=1):
        print(f"\n[{i}] score={score:.1f}, category={doc.metadata.get('category')}")
        print(doc.page_content.replace("\n", " ")[:300])


=== Hybrid 후보 (6개) ===

[1] category=onboarding_guide
### Q2: 재택근무는 어떻게 신청하나요? HR Portal에서 신청하며, 주 최대 2회까지 가능합니다. 팀장 사전 승인 필요.  ### Q3: 점심시간은 어떻게 되나요? 12:00-13:00이며, 구내식당 또는 외부 식당 이용 가능합니다.  ### Q4: 야근 시 

[2] category=faq
---  ## 2. 인사(HR) 및 복무 규정  ### Q4. 연차 및 특별 휴가 관리  * **잔여일 확인:** [HR Portal] → [내 정보] → [휴가/근태]에서 실시간 확인이 가능합니다. * **신청 기한:** 연차는 최소 3일 전 신청을 권장하며, 당일 

[3] category=onboarding_guide
---  ## 6. 주요 연락처  ### 6.1 지원 부서  | 부서 | 담당 업무 | 연락처 | |------|----------|--------| | **인사팀** | 근태, 복리후생, 급여 | hr@company.com / 내선 9012 | | **IT지원팀** 

[4] category=hr_policy
---  ## 5. 퇴직 및 오프보딩 절차  마지막까지 아름다운 마무리를 위해 원활한 인수인계를 지원합니다.  ### 5.1 퇴직 절차  * **사직서 제출:** 퇴직 희망일 기준 최소 **1개월 전**에 사직서를 제출하여 인력 충원 및 업무 승계 시간을 확보해야 합니

[5] category=hr_policy
유연하고 몰입도 높은 업무 환경을 위해 다음과 같은 근무 제도를 운영합니다.  ### 2.1 유연근무제 및 코어 타임  * **근무 시간:** 일 8시간(주 40시간) 근무를 원칙으로 합니다. * **선택적 근로제:** 08:00 ~ 10:00 사이 자유롭게 출근하되,

[6] category=expense_policy
### 6.2 구매 절차 1. ERP 구매 요청 등록 2. 승인권자 승인 3. 총무팀 구매 진행 (또는 직접 구매)

## 10. Query 변형으로 검색 질의 개선
- 사용자 질문은 항상 검색에 좋은 형태가 아님
- 예를 들어 "그거 어디서 승인받아?" 같은 질문은 문서의 실제 표현과 거리가 멂
- Query 변형은 LLM으로 질문을 더 검색 친화적인 표현으로 바꾼 뒤 retriever 에 넣는 방식
- 여기서는 간단한 **Multi-Query** 방식을 직접 구현
    - 한 질문을 여러 검색 질의로 확장하고, 결과를 중복 제거해 합침

### 10.1. Multi-Query Retriever
- LLM 이 원 질문을 N 개 다른 표현으로 바꿔서 각각 검색 → 결과 합집합

In [38]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-5.4-mini")

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

results = mq_retriever.invoke('집에서 일하려면 뭐 해야 하나요?')

print(f"\n총 {len(results)} 개 문서:")
for d in results:
    print(f"  {d.page_content[:70]}")


총 16 개 문서:
  ### 3.3 Day 3 (수요일)
| 시간 | 내용 | 담당 |
|------|------|------|
| 09:00-12
  # [사내 가이드] 무엇이든 물어보세요! (FAQ)

이 가이드는 임직원 여러분의 원활한 회사 생활을 돕기 위해 제작되었습니다
  ---

## 4. 접대비

### 4.1 사용 기준

| 구분 | 1인당 한도 | 비고 |
|------|----------
  # [인사 정책 매뉴얼] 즐겁고 공정한 직장 문화를 위한 가이드

본 안내서는 우리 회사의 핵심 인사 원칙과 규정을 담고 있습
  ---

## 5. 퇴직 및 오프보딩 절차

마지막까지 아름다운 마무리를 위해 원활한 인수인계를 지원합니다.

### 5.1 
  **식비:**
- 1일 5만원 한도 (아침 1만원, 점심 1.5만원, 저녁 2.5만원)
- 접대가 포함된 경우 중복 청구 불가
  ### 4.2 테스트 작성 원칙

**AAA 패턴:**
```python
def test_user_login():
    # 
  ### 1.3 경조사 휴가 상세

| 경조 항목 | 휴가 일수 | 비고 |
| --- | --- | --- |
| **본인 결
  ---

## 6. 주요 연락처

### 6.1 지원 부서

| 부서 | 담당 업무 | 연락처 |
|------|-------
  ---

## 2. 인사(HR) 및 복무 규정

### Q4. 연차 및 특별 휴가 관리

* **잔여일 확인:** [HR Po
  ### Q2: 재택근무는 어떻게 신청하나요?
HR Portal에서 신청하며, 주 최대 2회까지 가능합니다. 팀장 사전 승인 필
  # 신입사원 온보딩 가이드

새로운 가족이 되신 것을 환영합니다! 본 가이드는 원활한 적응과 빠른 업무 파악을 위해 작성되었습
  유연하고 몰입도 높은 업무 환경을 위해 다음과 같은 근무 제도를 운영합니다.

### 2.1 유연근무제 및 코어 타임

* *
  ### 3.3 Day 3 (수요일)
| 시간 | 내용 | 담당 |


### 10.2. HyDE (Hypothetical Document Embeddings)
- "질문" 보다 "답 문서" 가 vectorstore 안 문서와 더 비슷할 거라는 직관
- LLM 이 가상 답을 먼저 생성 → 그걸로 검색

In [39]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "주어진 질문에 대한 간단한 답을 1~2 문장으로 작성. 자료가 부족해도 그럴듯하게."),
    ("user", "{question}"),
])

hyde_chain = HYDE_PROMPT | llm | StrOutputParser()


def hyde_search(question: str, k: int = 3):
    hypothetical = hyde_chain.invoke({"question": question})
    print(f"가상 답: {hypothetical[:80]}")
    # 가상 답을 쿼리로 사용
    return retriever.invoke(hypothetical)

question = '집에서 일하려면 뭐 해야 하나요?'
print(f"질문: {question}")

results = hyde_search(question)
print("\n=== HyDE 검색 결과 ===")
for d in results:
    print(f"  {d.page_content[:70]}")

질문: 집에서 일하려면 뭐 해야 하나요?
가상 답: 집에서 일하려면 우선 원격으로 가능한 일을 찾고, 노트북·인터넷·업무용 환경을 갖추면 됩니다. 보통은 온라인 채용 사이트나 프리랜서 플랫폼에서 

=== HyDE 검색 결과 ===
  유연하고 몰입도 높은 업무 환경을 위해 다음과 같은 근무 제도를 운영합니다.

### 2.1 유연근무제 및 코어 타임

* *
  # [사내 가이드] 무엇이든 물어보세요! (FAQ)

이 가이드는 임직원 여러분의 원활한 회사 생활을 돕기 위해 제작되었습니다
  ---

## 4. 필수 시스템 및 도구

### 4.1 커뮤니케이션
| 도구 | 용도 | 접속 방법 |
|------|---
  ### Q6. 경조사 지원 및 서류 제출

* **지원 범위:** 본인 및 배우자, 직계존비속의 결혼, 환갑, 칠순, 사망 등
  ---

## 6. 주요 연락처

### 6.1 지원 부서

| 부서 | 담당 업무 | 연락처 |
|------|-------
  ### 1.3 경조사 휴가 상세

| 경조 항목 | 휴가 일수 | 비고 |
| --- | --- | --- |
| **본인 결
  **식비:**
- 1일 5만원 한도 (아침 1만원, 점심 1.5만원, 저녁 2.5만원)
- 접대가 포함된 경우 중복 청구 불가
  ---

## 2. 계정 설정 및 조직 구성

### 2.1 첫 시작: 계정 생성 및 인증

1. **접속:** [SmartW
  ---

## 5. 퇴직 및 오프보딩 절차

마지막까지 아름다운 마무리를 위해 원활한 인수인계를 지원합니다.

### 5.1 
  ### Q2. 사내 Wi-Fi 접속 및 보안 정책

* **업무용 Wi-Fi:** `COMPANY-OFFICE` (사내 보안 


## 11. 개선된 retriever 를 RAG 체인에 연결

- Reranker가 준비되어 있으면 `rerank_retriever` 를 사용하고, 없으면 `hybrid_retriever` 를 사용
- 검색기만 바꿔도 RAG 체인의 나머지 구조는 그대로 유지할 수 있음

In [40]:
# LCEL 체인에서 사용자의 입력 질문을 그대로 다음 단계로 넘기기 위한 Runnable
from langchain_core.runnables import RunnablePassthrough

# 검색된 Document 리스트를 LLM 프롬프트에 넣기 좋은 문자열로 변환하는 함수
def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "검색된 참고 자료가 없습니다."

    formatted = []
    for i, doc in enumerate(docs, start=1):
        formatted.append(
            f"[{i}] category={doc.metadata.get('category')}, source={doc.metadata.get('source')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [41]:
best_retriever = rerank_retriever or hybrid_retriever

rag_prompt = ChatPromptTemplate.from_messages([
    (
        'system',
        '당신은 회사 정책 안내 assistant입니다. 참고 자료에 있는 내용만 근거로 답하세요.'
        '''근거가 부족하면 '문서에서 해당 정보를 찾지 못햇습니다.'라고 답하세요.'''
    ),
    (
        'user', '참고 자료: \n{context}\n\n질문: {question}'
    )
])

llm = init_chat_model("openai:gpt-5.4-mini")

rag_chain = (
    {
        'context':best_retriever | format_docs,
        'question':RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = '교육비 지원 한도는 얼마인가요?'
print(rag_chain.invoke(question))

문서에서 확인되는 교육비 지원 한도는 **연간 200만 원**입니다.  
근거:
- 자기계발 및 교육(L&D): 외부 강의, 컨퍼런스, 대학원 학비 등을 **연간 200만 원** 한도로 지원
- FAQ: 직무 관련 학원, 온라인 강의, 도서 구입비, 외국어 시험 응시료를 **연간 200만 원 한도** 내에서 지원


## 12. 정리

- BM25는 정확한 키워드, 고유명사, 코드, 약어 검색에 강합니다.
- 벡터 검색은 표현이 달라도 의미가 가까운 문서를 찾는 데 강합니다.
- 하이브리드 검색은 BM25와 벡터 검색을 RRF 기반으로 합쳐 1차 후보 품질을 높입니다.
- Reranker는 후보 문서를 질문과 직접 비교해 2차로 재정렬합니다.
- Query 변형은 사용자의 모호한 질문을 검색 친화적인 여러 질의로 확장합니다.
- 실무 기본 패턴은 `Query 변형 → Hybrid 후보 검색 → Rerank → RAG 답변` 입니다.

## [실습]

1. `bm25_retriever.k`, `vector_retriever`의 `k` 값을 3, 5, 10으로 바꾸고 결과를 비교합니다.
2. `weights=[0.7, 0.3]`, `[0.5, 0.5]`, `[0.3, 0.7]` 로 같은 질문 5개를 검색해 어떤 질문에서 차이가 큰지 정리합니다.
3. `korean_tokenizer`에서 태그 집합을 바꿔 BM25 결과가 어떻게 달라지는지 확인합니다.
4. Cohere Rerank의 `top_n`을 1, 3, 5로 바꿔 RAG 답변 차이를 비교합니다.
5. Query 변형 + BM25 + 하이브리드 + Reranker 4단계를 한 chain 으로 묶고 RAG 답변을 확인합니다.